# Partes E e F · Aprender sem rótulos e prever números

**Workshop de Estudantes 2 · Projecto EYES TO EARTH (ETE) · 21 de Novembro de 2026**

Nas Partes B a D treinámos uma **classificação supervisionada**: exemplos com rótulos, resposta = uma categoria.
Agora vamos treinar os outros dois tipos de modelo que vimos na teoria:

| Parte | Tipo de aprendizagem | Pergunta | Tempo |
| --- | --- | --- | --- |
| E | **Não supervisionada** (agrupamento, K-means) | Sem nenhum rótulo, o computador consegue separar a água sozinho? | 15 min |
| F | **Supervisionada, regressão** | Conseguimos prever o **caudal** do rio daqui a 3 dias, um número em m³/s? | 15 min |


| Sinal | Quer dizer |
| --- | --- |
| ▶️ | Corra a célula: clique nela e prima **Shift + Enter** |
| ✏️ | Edite: troque os `...` pela sua resposta e corra outra vez |
| ✍️ | Responda por escrito, na célula de texto logo abaixo (duplo clique para escrever) |
| ✅ | A célula verifica a sua resposta |


**Como trabalhar no Jupyter Notebook:** clique numa célula e prima **Shift + Enter** para a correr (sempre por
ordem, de cima para baixo). Enquanto corre aparece `[*]` à esquerda; quando acaba, aparece um número. Para
escrever uma resposta, faça duplo clique na célula de texto. Guarde com **Ctrl + S**.

## As ferramentas (bibliotecas) que vamos usar

| Biblioteca | Para que serve |
| --- | --- |
| `numpy` (`np`), `matplotlib` (`plt`) | Contas com tabelas de números e gráficos (como no notebook 2) |
| `scikit-learn` (`sklearn`) | `KMeans` (agrupar), `LinearRegression` (a recta) e `DecisionTreeRegressor` (árvore que dá um número) |
| `pandas` (`pd`) | Tabelas com datas, como uma folha de Excel: os 20 anos de caudal e chuva |

## Célula 0 · Ligar as ferramentas ▶️

A mesma Célula 0 do notebook 2. Corra sem mudar nada.

In [ ]:
# Célula 0 · ferramentas da aula: corra sem mudar nada ▶️
import os                                   # os: encontrar ficheiros no computador
import json                                 # json: ler a lista de pontos da equipa
import numpy as np                          # numpy: contas com tabelas de números (uma imagem é uma tabela)
import matplotlib.pyplot as plt             # matplotlib: desenhar mapas e gráficos
from sklearn.ensemble import RandomForestClassifier   # scikit-learn: o modelo Random Forest
from sklearn import metrics                 # scikit-learn: as notas do modelo (acerto, IoU, F1, Kappa, ROC-AUC)

BANDAS = ["azul", "verde", "vermelho", "nir", "swir"]
INDICES = ["ndwi", "mndwi", "ndvi"]
TERRENO = ["hand", "altitude", "declive"]
RAIO = 2                                    # cada ponto marcado vale um quadrado de 5 x 5 píxeis (150 m x 150 m)


def encontrar(nome):
    """Procura um ficheiro de dados na pasta dados/ (ao lado do notebook, ou uma pasta acima)."""
    for pasta in ["dados", os.path.join("..", "dados"), "."]:
        caminho = os.path.join(pasta, nome)
        if os.path.exists(caminho):
            return caminho
    raise FileNotFoundError(f"Não encontrei {nome}. Abra o Jupyter na pasta Workshop_2/2_pratica: "
                            "os dados estão em 2_pratica/dados.")


def carregar(nome):
    """Abre uma imagem (.npz): as 5 bandas em reflectância (0 a 1) e os índices NDWI, MNDWI e NDVI."""
    z = np.load(encontrar(nome))
    img = {banda: z[banda] / 10000 for banda in BANDAS}
    with np.errstate(divide="ignore", invalid="ignore"):
        img["ndwi"] = (img["verde"] - img["nir"]) / (img["verde"] + img["nir"])
        img["mndwi"] = (img["verde"] - img["swir"]) / (img["verde"] + img["swir"])
        img["ndvi"] = (img["nir"] - img["vermelho"]) / (img["nir"] + img["vermelho"])
    img["nuvem"] = z["nuvem"]                   # True onde há nuvem, sombra ou falta de dados
    img["zona"] = z["zona"]                     # True dentro do distrito de Marracuene, fora do mar
    for chave in ["data", "satelite", "epoca"]:
        img[chave] = str(z[chave])
    img["caudal_m3s"] = float(z["caudal_m3s"])
    return img


def juntar_terreno(img, extra):
    """Acrescenta as camadas de terreno (iguais em todas as datas) às características da imagem."""
    for chave in TERRENO:
        img[chave] = extra[chave]
    return img


def cor_natural(img, brilho=0.15):
    """Junta vermelho, verde e azul numa imagem a cores."""
    return np.clip(np.dstack([img["vermelho"], img["verde"], img["azul"]]) / brilho, 0, 1)


def ver_imagem(img, titulo="", agua=(), nao_agua=(), grelha=100):
    """Mostra a imagem a cores com linhas e colunas numeradas, e os pontos marcados (círculo = água, quadrado = não água)."""
    plt.figure(figsize=(9, 12))
    plt.imshow(cor_natural(img))
    for linha, coluna in agua:
        plt.plot(coluna, linha, "o", ms=9, mfc="cyan", mec="black")
    for linha, coluna in nao_agua:
        plt.plot(coluna, linha, "s", ms=9, mfc="orange", mec="black")
    plt.xticks(range(0, img["verde"].shape[1], grelha))
    plt.yticks(range(0, img["verde"].shape[0], grelha))
    plt.grid(alpha=0.5, color="white")
    plt.xlabel("coluna")
    plt.ylabel("linha")
    plt.title(titulo)
    plt.show()


def ver_zona(img, linha, coluna, tamanho=60):
    """Lupa: mostra uma zona pequena à volta de (linha, coluna), com grelha fina, para escolher pontos certos."""
    l0, c0 = max(linha - tamanho, 0), max(coluna - tamanho, 0)
    recorte = cor_natural(img)[l0:linha + tamanho, c0:coluna + tamanho]
    plt.figure(figsize=(7, 7))
    plt.imshow(recorte, extent=(c0 - 0.5, c0 + recorte.shape[1] - 0.5, l0 + recorte.shape[0] - 0.5, l0 - 0.5))
    plt.plot(coluna, linha, "+", ms=20, color="red")
    plt.grid(alpha=0.5, color="white")
    plt.xlabel("coluna")
    plt.ylabel("linha")
    plt.title(f"Lupa à volta da linha {linha}, coluna {coluna}")
    plt.show()


def tabela(img, caracteristicas):
    """Uma linha por píxel e uma coluna por característica: é assim que o modelo lê a imagem."""
    pilha = np.stack([img[c] for c in caracteristicas], axis=-1)
    return np.nan_to_num(pilha.reshape(-1, len(caracteristicas)))


def exemplos(img, agua, nao_agua, caracteristicas, raio=RAIO, grupos=False):
    """Transforma os pontos marcados em exemplos: X (as características) e y (1 = água, 0 = não água).
    raio: cada ponto vale um quadrado de (2 x raio + 1)² píxeis. grupos=True devolve também o número do ponto
    de cada píxel (para a validação cruzada por pontos, no nível avançado)."""
    pilha = np.nan_to_num(np.stack([img[c] for c in caracteristicas], axis=-1))
    X, y, g, nas_nuvens = [], [], [], 0
    for classe, pontos in ((1, agua), (0, nao_agua)):
        for linha, coluna in pontos:
            janela = (slice(max(linha - raio, 0), linha + raio + 1), slice(max(coluna - raio, 0), coluna + raio + 1))
            limpo = ~img["nuvem"][janela]
            if not limpo.any():
                nas_nuvens += 1
                continue
            X.append(pilha[janela][limpo])
            y.append(np.full(limpo.sum(), classe))
            g.append(np.full(limpo.sum(), len(g)))
    if nas_nuvens:
        print(f"Atenção: {nas_nuvens} ponto(s) caíram em nuvens ou fora da imagem e foram ignorados.")
    if grupos:
        return np.concatenate(X), np.concatenate(y), np.concatenate(g)
    return np.concatenate(X), np.concatenate(y)


def prever(modelo, img, caracteristicas):
    """A probabilidade de ser água (0 a 1), para cada píxel da imagem."""
    probabilidade = modelo.predict_proba(tabela(img, caracteristicas))[:, 1]
    return probabilidade.reshape(img["verde"].shape)


def notas(verdade, previsto, probabilidade=None):
    """As notas do modelo comparado com a verdade (1 = água, 0 = não água)."""
    r = {"acerto": metrics.accuracy_score(verdade, previsto),
         "IoU": metrics.jaccard_score(verdade, previsto),
         "F1": metrics.f1_score(verdade, previsto),
         "Kappa": metrics.cohen_kappa_score(verdade, previsto)}
    if probabilidade is not None:
        r["ROC-AUC"] = metrics.roc_auc_score(verdade, probabilidade)
    return r


def mostrar_notas(nome, r):
    print(f"{nome:28s}", "  ".join(f"{k} {v:.3f}" for k, v in r.items()))


def contar_erros(verdade, previsto):
    """A matriz de confusão em palavras: o que acertou e o que errou."""
    agua_certa = int(((verdade == 1) & (previsto == 1)).sum())
    terra_certa = int(((verdade == 0) & (previsto == 0)).sum())
    falso_alarme = int(((verdade == 0) & (previsto == 1)).sum())
    agua_perdida = int(((verdade == 1) & (previsto == 0)).sum())
    print(f"Água que o modelo encontrou (certo):      {agua_certa:8d} píxeis")
    print(f"Terra que o modelo disse terra (certo):   {terra_certa:8d} píxeis")
    print(f"Falsos alarmes (terra que disse água):    {falso_alarme:8d} píxeis")
    print(f"Água perdida (água que disse terra):      {agua_perdida:8d} píxeis")
    return agua_certa, terra_certa, falso_alarme, agua_perdida


print("Ferramentas prontas.")

## ⚙️ Painel de configuração ▶️

Todas as escolhas destas duas partes estão aqui: o número de grupos do K-means, quantos dias à frente queremos
prever o caudal, as pistas da regressão, o ano que separa o treino do exame. Na primeira vez, corra sem mudar nada.
Para experimentar, mude um valor, corra esta célula e volte a correr as células da parte que quer ver.

In [ ]:
# ⚙️ PAINEL DE CONFIGURAÇÃO: mude os valores aqui e volte a correr as células da parte que quer ver
CONFIG = {
    # Parte E · o K-means (agrupamento, sem rótulos)
    "caracteristicas": BANDAS + INDICES,     # as pistas de cada píxel (pode juntar TERRENO)
    "k_grupos": 4,                           # quantos grupos (experimente 2, 3 ou 6)
    "inicializacoes": 3,                     # quantas vezes recomeça com centros diferentes (fica o melhor resultado)
    # Parte F · a regressão do caudal
    "dias_a_frente": 3,                      # prever o caudal daqui a quantos dias (experimente 1 ou 7)
    "pistas_caudal": ['caudal_m3s', 'chuva_7d_mm', 'chuva_14d_mm', 'chuva_30d_mm'],
    "ultimo_ano_de_treino": 2018,            # o treino vai até este ano; o exame é o que vem depois
    "profundidades": [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20],   # as árvores de regressão da curva do sobreajuste
    "semente": 0,                            # o mesmo número dá sempre o mesmo resultado
}
print("Configuração pronta:", ", ".join(f"{chave} = {valor}" for chave, valor in CONFIG.items() if chave != "caracteristicas"))

---
# Parte E · Agrupar sem rótulos: o K-means (15 min)

Na aprendizagem **não supervisionada** não há respostas certas. O computador recebe só as características
(`X`) e procura **grupos de píxeis parecidos**. O **K-means** faz assim:

1. espalha **K** centros ao acaso;
2. cada píxel junta-se ao centro mais perto (o mais parecido);
3. cada centro muda-se para o meio do seu grupo;
4. repete os passos 2 e 3 até os grupos não mudarem.

No fim, o computador sabe que os píxeis do grupo 1 são parecidos entre si, mas **não sabe que se chamam "água"**.
Somos nós que damos o nome aos grupos.

### Passo E1 · A tabela, sem rótulos ▶️

In [ ]:
cheia = carregar("incomati_2026_cheia.npz")
extra = dict(np.load(encontrar("camadas_extra.npz")))
juntar_terreno(cheia, extra)                           # o terreno fica disponível como pista (ver o painel ⚙️)
CARACTERISTICAS = CONFIG["caracteristicas"]

X = tabela(cheia, CARACTERISTICAS)                     # uma linha por píxel, uma coluna por característica
valido = (cheia["zona"] & ~cheia["nuvem"]).ravel()     # só os píxeis do distrito, sem nuvens
print("A tabela tem", X.shape[0], "píxeis e", X.shape[1], "características.")
print("Píxeis usados para aprender:", valido.sum())
print("Rótulos: nenhum! Ninguém disse ao computador o que é cada píxel.")

### Passo E2 · Treinar o K-means ✏️

Repare: `fit(X)` **sem** `y`. É isso que quer dizer "não supervisionada". O `StandardScaler` põe todas as
características na mesma escala, porque o K-means mede distâncias entre píxeis.

In [ ]:
from sklearn.cluster import KMeans                     # o K-means: junta os píxeis parecidos em K grupos
from sklearn.preprocessing import StandardScaler       # pôr todas as características na mesma escala
from sklearn.pipeline import make_pipeline             # encadear os dois passos num só modelo

K = CONFIG["k_grupos"]                                 # ✏️ quantos grupos? Mude no painel ⚙️ (experimente 2, 3 ou 6)
agrupador = make_pipeline(StandardScaler(), KMeans(n_clusters=K, n_init=CONFIG["inicializacoes"],
                                                   random_state=CONFIG["semente"]))
agrupador.fit(X[valido])                               # APRENDER SEM RÓTULOS: só há X, não há y

grupos = agrupador.predict(X).reshape(cheia["verde"].shape)      # o número do grupo de cada píxel
contagem = np.bincount(grupos.ravel()[valido], minlength=K)
for g in range(K):
    print(f"Grupo {g}: {100 * contagem[g] / contagem.sum():4.1f}% dos píxeis")

### Passo E3 · O mapa dos grupos ▶️

Cada cor é um grupo. Compare com a cor natural: que grupo parece ser a água? E a vegetação? E a areia?

In [ ]:
from matplotlib.colors import ListedColormap

cores = ListedColormap(plt.get_cmap("tab10").colors[:K])                   # uma cor por grupo
mapa_grupos = np.where(cheia["zona"] & ~cheia["nuvem"], grupos, np.nan)     # sem nuvens nem fora do distrito

fig, eixos = plt.subplots(1, 2, figsize=(14, 9))
eixos[0].imshow(cor_natural(cheia))
eixos[0].set_title("Cor natural, 24 jan 2026")
figura = eixos[1].imshow(mapa_grupos, cmap=cores, vmin=-0.5, vmax=K - 0.5, interpolation="nearest")
eixos[1].set_title(f"Os {K} grupos do K-means (sem rótulos)")
fig.colorbar(figura, ax=eixos[1], ticks=range(K), shrink=0.6, label="número do grupo")
for e in eixos:
    e.axis("off")
plt.tight_layout()
plt.show()

### Passo E4 · Dar nome aos grupos ✏️

O computador não sabe os nomes. **Vocês** decidem que grupo é a água, olhando para o mapa. Se a água ficou em
dois grupos (por exemplo, a água barrenta e a água escura), escrevam os dois: `[0, 3]`.

In [ ]:
GRUPOS_AGUA = None          # ✏️ escrevam o número do grupo da água, por exemplo [1]

g_validos = grupos.ravel()[valido]
mndwi_validos = cheia["mndwi"].ravel()[valido]          # o MNDWI só ajuda a dar nome aos grupos (o K-means não o precisa)
mndwi_medio = [np.nanmean(mndwi_validos[g_validos == g]) for g in range(K)]
for g in range(K):
    print(f"Grupo {g}: MNDWI médio {mndwi_medio[g]:+.2f}")
if GRUPOS_AGUA is None:
    GRUPOS_AGUA = [g for g in range(K) if mndwi_medio[g] > 0]
    print(f"\n✏️ Ainda não escolheram. Por agora uso a regra do Workshop 1 (MNDWI médio maior do que 0): {GRUPOS_AGUA}")
print("Grupos a que chamamos 'água':", GRUPOS_AGUA)

### Passo E5 · O exame: o K-means encontrou a água? ▶️

O mesmo exame do notebook 2: as amostras do Copernicus EMS. O K-means nunca viu um rótulo, nem do exame nem nosso.

In [ ]:
ems = extra["ems_2026"]                                 # 1 água, 0 não água, -1 sem rótulo
teste = (ems >= 0) & ~cheia["nuvem"]
verdade = ems[teste]
agua_kmeans = np.isin(grupos, GRUPOS_AGUA)

mostrar_notas("K-means (sem rótulos)", notas(verdade, agua_kmeans[teste].astype(int)))
mostrar_notas("Regra MNDWI > 0 (Workshop 1)", notas(verdade, (cheia["mndwi"][teste] > 0).astype(int)))
r = notas(verdade, agua_kmeans[teste].astype(int))
area = (agua_kmeans & cheia["zona"] & ~cheia["nuvem"]).sum() * 0.09
print(f"\nQUADRO DA TURMA -> K-means | K = {K} | grupos de água {GRUPOS_AGUA} | IoU {r['IoU']:.3f} | {area:.0f} ha de água")

plt.figure(figsize=(7, 10))
plt.imshow(agua_kmeans & cheia["zona"] & ~cheia["nuvem"], cmap="Blues")
plt.title(f"A água segundo o K-means: {area:.0f} ha")
plt.axis("off")
plt.show()

✍️ **Perguntas:** o K-means nunca viu um rótulo. Como é que conseguiu separar a água? Quem lhe deu o nome "água"?
Experimentem `"k_grupos": 2`, `3` e `6` no painel ⚙️ e corram outra vez do E2 ao E5: o que muda?

*Resposta:*

✍️ **Pergunta:** que **vantagem** e que **desvantagem** tem aprender sem rótulos?

*Resposta:*

---
# Parte F · Prever um número: o caudal daqui a 3 dias (15 min)

Uma **regressão** é aprendizagem supervisionada em que a resposta é um **número**. Aqui: o caudal do Incomáti
em Marracuene daqui a 3 dias, em m³/s. As pistas são o caudal de hoje e a chuva na bacia nos últimos
7, 14 e 30 dias. Os rótulos vêm... do futuro: o caudal que o rio teve, de facto, 3 dias depois.

Os dados (pasta `dados/`): caudal diário do **GloFAS** (Copernicus) e chuva diária na bacia do **CHIRPS**, de 2006 a 2026.

### Passo F1 · Vinte anos de caudal e chuva ▶️

In [ ]:
import pandas as pd                                    # pandas: tabelas com datas, como uma folha de Excel

serie = pd.read_csv(encontrar("caudal_e_chuva_diarios.csv"), parse_dates=["data"])
DIAS = CONFIG["dias_a_frente"]                         # prever o caudal daqui a DIAS dias (mude no painel ⚙️)
serie["caudal_futuro"] = serie["caudal_m3s"].shift(-DIAS)       # o rótulo: o caudal que o rio teve DIAS dias depois
serie = serie.dropna()
print(len(serie), "dias, de", serie["data"].min().date(), "a", serie["data"].max().date())

fig, eixos = plt.subplots(2, 1, figsize=(14, 6), sharex=True)   # dois gráficos, um por cima do outro
eixos[0].plot(serie["data"], serie["caudal_m3s"], color="steelblue", lw=0.8)
eixos[0].set_ylabel("Caudal (m³/s)")
eixos[0].set_title("O Incomáti em Marracuene: o caudal (em cima) e a chuva na bacia (em baixo)")
eixos[1].plot(serie["data"], serie["chuva_7d_mm"], color="gray", lw=0.8)
eixos[1].set_ylabel("Chuva de 7 dias (mm)")
plt.tight_layout()
plt.show()
serie.tail()

### Passo F2 · Treino e exame: o exame é o futuro ✏️

Quando o tempo importa, **não** se escolhem dias ao acaso para o exame: um dia e o dia seguinte são quase
iguais, e seria como copiar. Treinamos com o passado (2006 a 2018) e fazemos o exame com o futuro
(2019 a 2026), que o modelo nunca viu. Inclui a grande cheia de janeiro de 2026.

In [ ]:
PISTAS = CONFIG["pistas_caudal"]                       # ✏️ mude no painel ⚙️ (experimente tirar "caudal_m3s")
ULTIMO = CONFIG["ultimo_ano_de_treino"]
treino = serie[serie["data"].dt.year <= ULTIMO]        # aprender com o passado
exame = serie[serie["data"].dt.year > ULTIMO]          # o exame é o FUTURO
print(f"Treino: {len(treino)} dias (até {ULTIMO}) | Exame: {len(exame)} dias (de {ULTIMO + 1} em diante)")
print(f"Maior caudal no treino: {treino['caudal_m3s'].max():.0f} m³/s | no exame: {exame['caudal_m3s'].max():.0f} m³/s")

### Passo F3 · A regressão linear: a equação que melhor passa pelos pontos ▶️

O modelo aprende um número para cada pista (o **coeficiente**). As notas de uma regressão são diferentes das da
classificação:
- **erro médio (MAE)**: em média, quantos m³/s a previsão erra (quanto menor, melhor);
- **R²**: quanto do sobe e desce do rio o modelo explica (1 = tudo, 0 = nada).

In [ ]:
from sklearn.linear_model import LinearRegression      # regressão linear: a recta (ou o plano) que melhor passa pelos pontos

recta = LinearRegression()
recta.fit(treino[PISTAS], treino["caudal_futuro"])      # APRENDER: X = as pistas, y = o caudal DIAS dias depois

print("A equação que o modelo aprendeu (o sinal + ou - diz se a pista faz subir ou descer a previsão):")
equacao = " ".join(f"{c:+.2f} × {p}" for c, p in zip(recta.coef_, PISTAS))   # :+ mostra sempre o sinal
print(f"  caudal daqui a {DIAS} dias = {equacao} {recta.intercept_:+.1f}")

previsto = recta.predict(exame[PISTAS])                 # PREVER os dias do exame
erro_medio = metrics.mean_absolute_error(exame["caudal_futuro"], previsto)
r2 = metrics.r2_score(exame["caudal_futuro"], previsto)
igual_a_hoje = metrics.mean_absolute_error(exame["caudal_futuro"], exame["caudal_m3s"])
print(f"\nNo exame: erro médio (MAE) {erro_medio:.0f} m³/s | R² {r2:.2f}")
print(f"Para comparar, dizer 'daqui a {DIAS} dias fica igual a hoje' erra em média {igual_a_hoje:.0f} m³/s")

### Passo F4 · O exame em gráficos ▶️

À esquerda, a cheia de janeiro de 2026: o que aconteceu e o que o modelo tinha previsto DIAS dias antes, com os
limites de alerta. À direita, todos os dias do exame: se o modelo fosse perfeito, os pontos ficavam em cima da linha.

In [ ]:
janela = ((exame["data"] >= "2025-12-20") & (exame["data"] <= "2026-02-15")).to_numpy()
dia_previsto = exame["data"][janela] + pd.Timedelta(days=DIAS)          # o dia a que a previsão se refere

fig, eixos = plt.subplots(1, 2, figsize=(16, 6))
eixos[0].plot(dia_previsto, exame["caudal_futuro"][janela], color="black", lw=2, label="o que aconteceu")
eixos[0].plot(dia_previsto, previsto[janela], color="tab:blue", lw=2, label=f"o que o modelo previu {DIAS} dias antes")
for limite, nome, cor in [(164, "amarelo", "goldenrod"), (376, "laranja", "darkorange"), (970, "vermelho", "firebrick")]:
    eixos[0].axhline(limite, color=cor, ls="--", lw=1)
    eixos[0].text(dia_previsto.iloc[0], limite + 40, f"alerta {nome}", color=cor)
eixos[0].set_ylabel("Caudal (m³/s)")
eixos[0].set_title("A cheia de janeiro de 2026")
eixos[0].legend(loc="upper right")
fig.autofmt_xdate()

eixos[1].scatter(exame["caudal_futuro"], previsto, s=6, alpha=0.4)
maximo = exame["caudal_futuro"].max()
eixos[1].plot([0, maximo], [0, maximo], color="black", lw=1, label="previsão perfeita")
eixos[1].set_xlabel("Caudal que aconteceu (m³/s)")
eixos[1].set_ylabel("Caudal previsto (m³/s)")
eixos[1].set_title(f"Todos os dias do exame: erro médio {erro_medio:.0f} m³/s")
eixos[1].legend()
plt.tight_layout()
plt.show()

### Passo F5 · Uma árvore de regressão e o sobreajuste ▶️

Uma árvore também pode dar um número: em vez de "água" ou "terra", cada folha tem um caudal. Quanto mais
**profunda** (mais perguntas seguidas), mais o modelo se ajusta ao treino. Vamos desenhar o gráfico clássico
da teoria, com os nossos dados: a nota no treino e a nota no exame, para cada profundidade.

In [ ]:
from sklearn.tree import DecisionTreeRegressor         # árvore de regressão: perguntas de sim ou não que acabam num número

profundidades = CONFIG["profundidades"]                # as árvores da curva (mude no painel ⚙️)
nota_treino, nota_exame = [], []
for prof in profundidades:
    arvore = DecisionTreeRegressor(max_depth=prof, random_state=CONFIG["semente"]).fit(treino[PISTAS], treino["caudal_futuro"])
    nota_treino.append(metrics.r2_score(treino["caudal_futuro"], arvore.predict(treino[PISTAS])))
    nota_exame.append(metrics.r2_score(exame["caudal_futuro"], arvore.predict(exame[PISTAS])))

plt.figure(figsize=(9, 5))
plt.plot(profundidades, nota_treino, "o-", label="nota no treino (o que já viu)")
plt.plot(profundidades, nota_exame, "o-", label="nota no exame (o futuro)")
plt.axhline(r2, color="gray", ls=":", label=f"regressão linear no exame ({r2:.2f})")
plt.xlabel("Profundidade da árvore (quantas perguntas seguidas)")
plt.ylabel("R² (1 = perfeito)")
plt.title("Sobreajuste: a nota do treino sobe sempre, a do exame não")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

melhor = profundidades[int(np.argmax(nota_exame))]
print(f"Melhor profundidade no exame: {melhor} (R² {max(nota_exame):.2f}). "
      f"Com 20 perguntas: treino {nota_treino[-1]:.2f}, exame {nota_exame[-1]:.2f}.")
print(f"\nQUADRO DA TURMA -> regressão | erro médio da recta {erro_medio:.0f} m³/s | R² {r2:.2f} | melhor árvore: profundidade {melhor}")

✍️ **Pergunta:** a recta previu a subida da cheia de janeiro de 2026 a tempo de dar um alerta laranja? Porquê?

*Resposta:*

✍️ **Pergunta:** no gráfico do Passo F5, a partir de que profundidade a árvore começa a **decorar** o treino?
Porque é que a nota do exame desce?

*Resposta:*

---
## O que aprendemos

1. **Não supervisionada:** o K-means encontra **grupos** de píxeis parecidos, sem rótulos. O nome dos grupos
   é dado por uma pessoa.
2. **Classificação** dá uma categoria ("água"); **regressão** dá um número (o caudal em m³/s).
3. Na regressão, as notas são o **erro médio (MAE)** e o **R²**.
4. Quando o tempo importa, **o exame é o futuro**: treina-se com o passado e testa-se com anos que o modelo nunca viu.
5. Mais complexo não é sempre melhor: uma árvore muito profunda **decora** o treino (sobreajuste) e erra mais no exame.

**Por último:** a Parte G, com a ficha `4_ficha_da_plataforma.md`, e a plataforma EYES TO EARTH.

## Se aparecer um erro

Leia **a última linha** da mensagem vermelha. Quase sempre é um destes:

| Última linha do erro | O que quer dizer | O que fazer |
| --- | --- | --- |
| `NameError: name 'np' is not defined` | Saltou a Célula 0 | Corra a Célula 0 e depois as outras, por ordem |
| `NameError: name 'cheia' is not defined` | Saltou a célula que carrega os dados | Corra as células por ordem, de cima para baixo |
| `FileNotFoundError: Não encontrei ...` | O Jupyter não foi aberto na pasta da prática | Abra o notebook a partir da pasta `Workshop_2/2_pratica` (os dados estão em `2_pratica/dados`) |
| `ModuleNotFoundError: No module named 'sklearn'` | Falta uma biblioteca neste computador | Chame o Facilitador 6 (`pip install scikit-learn`) |
| `ValueError: need at least one array to concatenate` | Não há pontos, ou caíram todos em nuvens | Marque pontos fora das nuvens, ou use os pontos da equipa |
| `SyntaxError` | Falta um parêntese, uma aspa ou uma vírgula | Compare a sua linha com o exemplo logo acima |
| `IndentationError` | Os espaços no início da linha estão errados | Use 4 espaços dentro de `if`, `for` e `def` |
| *Kernel died* / *Dead kernel* | O Python parou (memória ou erro grave) | Menu **Kernel → Restart Kernel** e corra outra vez por ordem |

Se nada resultar: menu **Kernel → Restart Kernel and Run All Cells** (nas versões antigas do Jupyter:
**Kernel → Restart & Run All**). O trabalho fica guardado no próprio computador: **Ctrl + S** para guardar.